In [ ]:
import os
import re
from collections import Counter, defaultdict

import nltk
import pandas as pd
from langchain_community.document_loaders import PyMuPDFLoader
from nltk.corpus import words

In [ ]:
DEFAULT_STOPWORDS = {
    "what",
    "why",
    "when",
    "where",
    "which",
    "who",
    "whom",
    "whose",
    "how",
    "the",
    "a",
    "an",
    "is",
    "are",
    "was",
    "were",
    "be",
    "being",
    "been",
    "to",
    "of",
    "in",
    "on",
    "for",
    "with",
    "and",
    "or",
    "but",
    "if",
    "then",
    "than",
    "by",
    "from",
    "as",
    "at",
    "it",
    "its",
    "that",
    "this",
    "these",
    "those",
    "do",
    "does",
    "did",
    "doing",
    "done",
    "so",
    "such",
    "into",
    "over",
    "under",
    "between",
    "within",
    "without",
    "about",
    "against",
    "can",
    "could",
    "should",
    "would",
    "may",
    "might",
    "must",
    "will",
    "shall",
    "i",
    "you",
    "he",
    "she",
    "we",
    "they",
    "them",
    "me",
    "my",
    "your",
    "our",
    "their",
    "any",
    "all",
    "each",
    "every",
    "both",
    "some",
    "no",
    "not",
    "nor",
    "also",
    "up",
    "down",
    "out",
    "off",
    "again",
    "further",
    "more",
    "most",
    "other",
    "own",
    "same",
    "because",
    "until",
    "while",
    "during",
    "before",
    "after",
    "what's",
    "whats",
    "it's",
    "im",
    "dont",
    "doesnt",
    "didnt",
    "cant",
    "couldnt",
    "shouldnt",
    "wont",
    "isnt",
    "arent",
    "wasnt",
    "werent",
    # finance filler
    "latest",
    "recent",
    "recently",
    "following",
    "below",
    "above",
    "document",
    "documents",
    "types",
    "type",
    "rank",
    "ranking",
    "indices",
    "index",
    # non related words
    "inc",
    "has",
    "one",
    "say",
    "due",
    "chunk",
    "chunks",
    "text",
}

In [ ]:
def extract_keywords(text: str, stopwords: dict = DEFAULT_STOPWORDS) -> list:
    """Extract keywords from text after removing stopwords."""
    words = re.findall(r"\b[a-z]{3,}\b", text.lower())
    return [w for w in words if w not in stopwords]


def get_pdf_text(pdf_path: str) -> str:
    """Load a PDF and return concatenated text from all pages."""
    loader = PyMuPDFLoader(pdf_path)
    pages = loader.load()
    return " ".join(page.page_content for page in pages)


def infer_doc_type(filename: str, doc_types: list[str]) -> str | None:
    """Infer document type from filename."""
    upper_name = filename.upper()
    for dt in doc_types:
        if dt.replace("-", "") in upper_name.replace("-", ""):
            return dt
    return None


def load_corpus_from_pdfs(pdf_dir: str, doc_types: list[str]) -> pd.DataFrame:
    """Load all PDFs in directory into a DataFrame with columns: doc_type, text."""
    records = []

    for fname in os.listdir(pdf_dir):
        if not fname.lower().endswith(".pdf"):
            continue

        doc_type = infer_doc_type(fname, doc_types)
        if doc_type is None:
            continue

        pdf_path = os.path.join(pdf_dir, fname)
        try:
            text = get_pdf_text(pdf_path)
        except Exception as e:
            print(f"[WARN] Failed to read {fname}: {e}")
            continue

        records.append({"doc_type": doc_type, "text": text})

    return pd.DataFrame(records)


def top_keywords_by_doc_type_share(
    df: pd.DataFrame,
    extract_keywords: callable,
    top_k: int = 6,
    min_global_count: int = 1,
    min_doc_type_count: int = 3,
    max_share: float = 99.9,
    validate_words: bool = True,
    min_word_length: int = 3,
) -> dict[str, list[tuple[str, float]]]:
    """Compute top keywords per document type based on.

        count(keyword in doc_type) / count(keyword in corpus)

    Filters out abbreviations, typos, and non-English words.
    """
    if validate_words:
        nltk.download("words")
        ENGLISH_WORDS = {w.lower() for w in words.words()}

    # ---- Global counts ----
    corpus_counts = Counter()
    for text in df["text"]:
        corpus_counts.update(extract_keywords(text))

    # ---- Per-doc-type counts ----
    doc_type_counts = defaultdict(Counter)
    for _, row in df.iterrows():
        doc_type_counts[row["doc_type"]].update(extract_keywords(row["text"]))

    # ---- Compute shares ----
    results = {}

    for doc_type, counts in doc_type_counts.items():
        keyword_shares = []

        for word, count in counts.items():
            global_count = corpus_counts[word]

            if global_count < min_global_count:
                continue
            if count < min_doc_type_count:
                continue

            if validate_words:
                word_lower = word.lower()
                if len(word_lower) < min_word_length:
                    continue
                if word_lower not in ENGLISH_WORDS:
                    continue

            share = count / global_count
            share_pct = share * 100

            if share_pct > max_share:
                continue

            keyword_shares.append((word, share_pct, count))

        results[doc_type] = [
            (word, share) for word, share, _ in sorted(keyword_shares, key=lambda x: (x[1], x[2]), reverse=True)[:top_k]
        ]

    return results

In [ ]:
PATH_PDFS = "../../../data/financebench/pdfs"
DOC_TYPES = ["DEF 14A", "10-K", "10-Q", "8-K", "EARNINGS"]
TOP_K = 10
MIN_GLOBAL_COUNT = 20

print("Loading PDFs...")
df = load_corpus_from_pdfs(PATH_PDFS, DOC_TYPES)

print(f"Loaded {len(df)} documents")

results = top_keywords_by_doc_type_share(
    df=df, extract_keywords=extract_keywords, top_k=TOP_K, min_global_count=MIN_GLOBAL_COUNT
)

for doc_type, keywords in results.items():
    print(f"\n{doc_type}:")
    for word, pct in keywords:
        print(f"  {word:20s}: {pct:6.2f}%")
